# Create artificial networks from scratch

In [ ]:
import numpy as np
import plotly.graph_objects as go

In [ ]:
## Choose the general theme for plotly.
# theme = 'plotly_white'
theme = "plotly_dark"

#### We start by creating two similar networks by distributing beads semi-randomly in a sphere
You might need to run this cell multiple times to get a result as it is not always possible to satisfy the constraints.

In [ ]:
from elastory.utils.network_tools import distribute_beads

# number of beads
N = 90

# maximal distance for beads from the center of mass
r_max = 12

# minimal distance between two neighbors
l_min = 2.2

# the cutoff length below which beads are connected by a spring
cutoff_length = 6

# the pockets could represent binding sites
pocket_0_center = np.array([r_max - l_min, 0, 0])
pocket_1_center = np.array([-r_max + l_min, 0, 0])
pocket_0_radius = cutoff_length * 1.2
pocket_1_radius = cutoff_length * 1.2

# generate initial positions
pos_init_0 = distribute_beads(
    N,
    r_max=r_max,
    l_min=l_min,
    max_attempts=int(1e3),
    cutoff_length=cutoff_length,
    p0_center=pocket_0_center,
    p1_center=pocket_1_center,
    p0_radius=pocket_0_radius,
    p1_radius=pocket_1_radius,
)

pos_init_1 = distribute_beads(
    N,
    r_max=r_max,
    l_min=l_min,
    max_attempts=int(1e3),
    cutoff_length=cutoff_length,
    p0_center=pocket_0_center,
    p1_center=pocket_1_center,
    p0_radius=pocket_0_radius,
    p1_radius=pocket_1_radius,
)

#### And merge them into one point-cloud by partly overlapping them and removing beads which are too close

In [ ]:
from elastory.utils.network_tools import merge_bead_positions

# the amount by which the two networks are shifted
shift = 0.1 * cutoff_length

# merge the two point clouds
pos_final = merge_bead_positions(pos_init_0, pos_init_1, shift=shift, l_min=l_min)

#### Instantiate a network object:

In [ ]:
from elastory.network.network import Network
from uuid import uuid4

net = Network(
    identifier=uuid4().hex,
    bead_positions=pos_final,
    cutoff_length=cutoff_length,
)

#### Some properties are directly available

In [ ]:
net.laplacian

In [ ]:
np.set_printoptions(precision=3)
net.hessian

In [ ]:
net.graph.graph.number_of_edges()

#### Plot the network in 3D

In [ ]:
from elastory.plot.plotly.utils import plot_network, update_layout

fig = go.Figure()
fig = plot_network(fig, net)
fig = update_layout(fig, hide_axes=False, show_legend=False, theme=theme)
fig.show()

### Choose source and target beads
I found it works best to choose them interactively in the plot. Just hover over the beads to get the indices. If it helps, visualize a convex hull.

In [ ]:
from elastory.plot.plotly.utils import plot_3d_convex_hull

fig = go.Figure()
fig = plot_network(fig, net)
fig = plot_3d_convex_hull(fig, points=net.geometry.bead_positions)
fig = update_layout(fig, theme=theme, show_legend=True)
fig.show()

In [ ]:
# just a default for source and target for running the notebook non-interactively

source = list(range(5))
target = [net.N - (i + 1) for i in range(5)]

In [ ]:
from elastory.utils.network_tools import estimate_pocket

source, source_pairs = estimate_pocket(source, net.geometry.bead_positions)

net.source = source
net.target = target
net.source_pairs = source_pairs

In [ ]:
from elastory.plot.plotly.utils import plot_network, update_layout

fig = go.Figure()
fig = plot_network(
    fig,
    net,
    labels=False,
)
fig = update_layout(fig, hide_axes=False, show_legend=False, theme=theme)
fig.show()

#### Save the network to a file

In [ ]:
from pathlib import Path

save_path = Path("./networks/")
save_path.mkdir(exist_ok=True)

file_path = save_path / f"{net.identifier}.yaml"
net.save_to_file(file_path)